In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import torchvision
from torchvision import datasets,models,transforms
from torch.utils.data import DataLoader
import zipfile
import os
import time
import shutil
import io
import pandas as pd

In [6]:
device=torch.device("cuda:0"if torch.cuda.is_available() else "cpu")
print(device)

#En colab establecer gpu como device

cpu


Se realiza el cambio de formato, se realiza el cambio de tamaño, y posteriormente se realiza el csv con los datos recopilados y las clases marcadas, una vez se tenga todo, se realiza un training con cierto porcentaje de la base de datos (80% training) y después una validación dentro de la misma (20% validation) y posteriormente realizar testing con muestras totalmente ajenas

Para python-colab, se tiene que realizar la base de datos en .zip, no otro tipo de compresión 

In [5]:
class AlexNet(nn.Module):
    def __init__(self,num_classes=3): #Perros,Gatos,Calzado
        super(AlexNet,self).__init__()

        #Capas Convolucionales
        
        #capa1
        self.conv1=nn.Conv2d(3,96,kernel_size=11,stride=4,padding=0)
        self.relu1=nn.ReLu(inplace=True)
        self.pool1=nn.MaxPool2d(kernel_size=3,stride=2)
        
        #capa2
        self.conv2=nn.Conv2d(96,256,kernel_size=5,stride=1,padding=2)
        self.relu2=nn.ReLu(inplace=True)
        self.pool2=nn.MaxPool2d(kernel_size=3,stride=2)
        
        #capa3
        self.conv3=nn.Conv2d(256,384,kernel_size=3,stride=1,padding=1)
        self.relu3=nn.ReLu(inplace=True)
        # self.pool3=nn.MaxPool2d(kernel_size=3,stride=2)
        
        #capa4
        self.conv4=nn.Conv2d(384,384,kernel_size=3,stride=1,padding=1)
        self.relu4=nn.ReLu(inplace=True)
        # self.pool4=nn.MaxPool2d(kernel_size=3,stride=2)
        
        #capa5
        self.conv5=nn.Conv2d(384,384,kernel_size=3,stride=1,padding=1)
        self.relu5=nn.ReLu(inplace=True)
        self.pool5=nn.MaxPool2d(kernel_size=3,stride=2)

        #Capas completamente conectadas

        #capa6
        self.fc6=nn.Linear(256*256,4096)
        self.relu6=nn.ReLu(inplace=True)
        self.drop6=nn.Dropout(p=0.5)

        #capa7
        self.fc7=nn.Linear(4096,4096)
        self.relu7=nn.ReLu(inplace=True)
        self.drop7=nn.Dropout(p=0.5)

        #capa8
        self.fc8=nn.Linear(4096,num_classes)
        # self.relu8=nn.ReLu(inplace=True)
        # self.drop8=nn.Dropout(p=0.5)

    def forward(self, x):
        # Bloque 1
        x = self.conv1(x)
        x = self.relu1(x)
        # x = self.lrn1(x)
        x = self.pool1(x)
 
        # Bloque 2
        x = self.conv2(x)
        x = self.relu2(x)
        # x = self.lrn2(x)
        x = self.pool2(x)
 
        # Bloque 3
        x = self.conv3(x)
        x = self.relu3(x)
 
        # Bloque 4
        x = self.conv4(x)
        x = self.relu4(x)
 
        # Bloque 5
        x = self.conv5(x)
        x = self.relu5(x)
        x = self.pool5(x)
 
        # Aplanar
        x = x.view(x.size(0), 256 * 6 * 6) #Revisar tamaño de salida con torch el resumen de la arquitectura, para evitar conflictos con view
 
        # FC6
        x = self.fc6(x)
        x = self.relu6(x)
        x = self.drop6(x)
 
        # FC7
        x = self.fc7(x)
        x = self.relu7(x)
        x = self.drop7(x)
 
        # FC8 (sin softmax, normalmente en el criterio de pérdida)
        x = self.fc8(x)

        return x

In [7]:
modelo=AlexNet()
modelo=modelo.to(device)

AttributeError: module 'torch.nn' has no attribute 'ReLu'

In [ ]:
#Adaptación para colab para subir el .zip de la base de datos
from google.colab import files
uploader=files.upload()

In [ ]:
datos=zipfile.ZipFile(io.BytesIO(uploader['Nombre del archivo zip']),'r')
datos.extractall("imagenes/")

In [ ]:
root= 'imagenes/archivo.csv'
img_list=os.listdir(root)
print(len(img_list))

basedatos=pd.read_csv('imagenes/archivo.csv')
basedatos=basedatos[['Nombre','Categoría']]

print(basedatos)

In [1]:
#Código para realizar la clasificación de la base de datos entre carpetas separadas
!rm -rf datos
!mkdir datos && mkdir datos/perros && mkdir datos/gatos && mkdir datos/zapatos

'rm' is not recognized as an internal or external command,
operable program or batch file.
The syntax of the command is incorrect.


In [ ]:
s0 = 0  # contador para gatos
s1 = 0  # contador para perros
s2 = 0  # contador para zapatos
num = 1 #longitud de img_list
 
for i, (_, i_row) in enumerate(basedatos.iterrows()):
    # Categoría 0 = Gatos
    if i_row['Categoria'] == 0 and s0 < num:
        s0 += 1
        shutil.copyfile('imagenes/basedatos/'+i_row['Nombre'], 'datos/gatos/'+i_row['Nombre']) #Ubicar la carpeta que contiene las imágenes de la base de datos y poner su dirección 
 
    # Categoría 1 = Perros
    elif i_row['Categoria'] == 1 and s1 < num:
        s1 += 1
        shutil.copyfile('imagenes/basedatos/'+i_row['Nombre'], 'datos/perros/'+i_row['Nombre'])
 
    # Categoría 2 = Zapatos
    elif i_row['Categoria'] == 2 and s2 < num:
        s2 += 1
        shutil.copyfile('imagenes/basedatos/'+i_row['Nombre'], 'datos/zapatos/'+i_row['Nombre'])
 
    # Verificar si ya se han copiado suficientes imágenes de cada categoría
    if s0 == num and s1 == num and s2 == num:
        break

##Se genera carpeta datos, y dentro de la misma se realizan las subcarpetas perros, gatos y zapatos

In [ ]:
# Obtener la lista de imágenes desde las 3 clases
img_list = os.listdir('datos/perros/')
img_list.extend(os.listdir('datos/gatos/'))
img_list.extend(os.listdir('datos/zapatos/'))
 
# Eliminar la carpeta ipynb_checkpoints si existe
!rm -rf 'datos/ipynb_checkpoints/'
 
# Definir transformaciones: conversión a tensor, resize y normalización
transformar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], #Promedio de rojos, verdes, y azules de todas las imágenes
                         std=[0.229, 0.224, 0.225]) #Desviación estadar
]) #Tensores normalizados con desviación estandar de 1
 
# Cargar dataset con ImageFolder
clases = datasets.ImageFolder('datos', transform=transformar)
 
# Separar el dataset en entrenamiento y test (80%-20%)
train_size = int(len(img_list)*0.8)
test_size = len(img_list) - train_size
entrenamiento, testeo = torch.utils.data.random_split(clases, [train_size, test_size])
 
# Crear dataloaders para batch y shuffle
cargar_entrenamiento = torch.utils.data.DataLoader(entrenamiento, batch_size=32, shuffle=True)
cargar_testeo = torch.utils.data.DataLoader(testeo, batch_size=32, shuffle=True)
 
print(cargar_testeo)

In [ ]:
#Muestra imágenes
# Obtener un batch de datos del cargador de entrenamiento
x, y = next(iter(cargar_entrenamiento))
 
# Crear una figura con una cuadrícula de 4x4 imágenes
fig, axs = plt.subplots(4, 4, figsize=(10,10))
 
# Iterar sobre los ejes creados en el grid
for i, ax in enumerate(axs.flatten()):
    # Permutar las dimensiones del tensor para (Alto, Ancho, Canales) y convertir a numpy
    pic = x[i].permute(1, 2, 0).numpy()
    
    # Normalizar la imagen para mostrarla adecuadamente
    pic = pic - pic.min()
    pic = pic / pic.max()
    
    # Obtener la etiqueta correspondiente a partir del índice
    label = clases.classes[y[i]]
    
    # Mostrar la imagen
    ax.imshow(pic)
    
    # Añadir el texto con la etiqueta en la esquina superior izquierda
    ax.text(0, 0, label, ha='left', va='top', fontweight='bold',
            color='k', backgroundcolor='y')
    
    # Ocultar los ejes
    ax.axis('off')
 
plt.tight_layout()
plt.show()

In [ ]:
#Para poder visualizar que es lo que posee el modelo realizado
torch.summary(modelo)

In [ ]:
#Líneas para correr AlexNet
# Definir el optimizador y el criterio de pérdida
optimizador = optim.Adam(modelo.parameters(), lr=0.0001)  # Adam: Gradiente adaptable
criterio = nn.CrossEntropyLoss()  # Función de pérdida de entropía cruzada
 
# Número de épocas de entrenamiento
epocas = 100
 
# Lista para almacenar el error promedio por época
errores_entrenamiento = []
 
for epoca in range(epocas):
    # Inicializar el error total por época
    error_total = 0
 
    # Entrenamiento (forward, backward, step)
    for idx, (imagenes, etiquetas) in enumerate(cargar_entrenamiento):
        # Mover datos a GPU si está disponible
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
 
        # Inicializar el gradiente a cero antes del forward-backward
        optimizador.zero_grad()
 
        # Paso forward: obtener predicciones del modelo
        prediccion = modelo(imagenes)
 
        # Calcular el error (pérdida) entre las predicciones y las etiquetas reales
        error = criterio(prediccion, etiquetas)
 
        # Acumular el error total
        error_total += error.item()
 
        # Backward: calcular gradientes
        error.backward()
 
        # Actualizar parámetros del modelo
        optimizador.step()
 
    # Calcular el error promedio de la época
    error_promedio = error_total / (idx + 1)
    errores_entrenamiento.append(error_promedio)
 
    # Imprimir el error de entrenamiento de la época actual
    print(f'Época: {epoca} | Error de entrenamiento: {error_promedio:.4f}')
 
    # Validación (o testeo) en cada época, sin cálculo de gradientes
    with torch.no_grad():
        error_testeo_total = 0
        contador_batch_testeo = 0
        for img_test, lbl_test in cargar_testeo:
            img_test, lbl_test = img_test.to(device), lbl_test.to(device)
            salida = modelo(img_test)
            error_testeo = criterio(salida, lbl_test)
            error_testeo_total += error_testeo.item()
            contador_batch_testeo += 1
 
        error_testeo_promedio = error_testeo_total / contador_batch_testeo
        print(f'Época: {epoca} | Error de test: {error_testeo_promedio:.4f}')
 
# Graficar la evolución del error de entrenamiento
plt.figure(figsize=(10,6))
plt.plot(errores_entrenamiento, label='Error de Entrenamiento')
plt.title('Evolución del Error de Entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Error')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Evaluación de la red
# Obtener un batch del cargador de test
xx, yy = next(iter(cargar_testeo))
 
# Mover imágenes y etiquetas a device (GPU si está disponible)
xx = xx.to(device)
yy = yy.to(device)
 
# Deshabilitar gradientes para la fase de test
with torch.no_grad():
    # Obtener las predicciones del modelo
    pre_test = torch.argmax(modelo(xx), dim=1)
 
# Crear figura con una cuadricula de 5x5 imágenes
fig, axs = plt.subplots(5, 5, figsize=(7,7))
 
for i, ax in enumerate(axs.flatten()):
    # Convertir imagen a formato numpy para visualizar
    # Primero permutar a (H, W, C) y luego pasar a CPU y numpy
    PIC = xx[i].permute(1,2,0).cpu().numpy()
    
    # Normalizar la imagen para visualizarla adecuadamente
    PIC = PIC - PIC.min()
    PIC = PIC / PIC.max()
    
    # Obtener la predicción y la etiqueta real
    prediccion_label = clases.classes[pre_test[i]]
    etiqueta_real = clases.classes[yy[i]]
    
    # Título con predicción y valor real
    title = f'Pred: {prediccion_label}\nTrue: {etiqueta_real}'
    color_title = 'green' if prediccion_label == etiqueta_real else 'red'
    
    # Mostrar la imagen y el título
    ax.imshow(PIC)
    ax.set_title(title, color=color_title, fontsize=8)
    ax.axis('off')
 
plt.tight_layout()
plt.show()